# FiberPulse AI: Day 1 reference notebook

**Date:** 2026-10-13
**Purpose:** a record of what we planned, decided, measured and built on day 1, with a few small runnable cells.

> Everything in this project is a learning build on **synthetic data**. Prices, margins and policies are invented.
> Keep it on a personal machine and separate from employer data.

**Contents:** 1) the system, 2) build plan, 3) decisions, 4) your hardware benchmark, 5) token budget,
6) setup recap, 7) Phase 0 code tour, 8) LLM client demo, 9) lessons from day 1, 10) next steps.

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains pyproject.toml) and make src/ importable
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))
print("Project root:", root)

## 1. The system we are building

A multi-agent B2B fiber Customer Value Management (CVM) system for SMB accounts, in three layers:

| Layer | Purpose | Components |
|---|---|---|
| 1. Sense | Predictive data infrastructure | Agent 1 External Data Enrichment, Agent 2 Voice Intelligence (ASR + PII masking), Databricks Lakehouse, DataRobot predictive engine (churn + acquisition, about 5,000 scores a month, top 2-3K flagged) |
| 2. Reason & Decide | LangGraph cognitive engine | Orchestrator, Agent 3 Researcher, Agent 4 Paradox Triage, Agent 5 Offer Architect, Agent 6 RAG Compliance, Agent 7 Prioritizer, Customer Outcomes KB (RAG), Exception Exit |
| 3. Act & Learn | Human verification, delivery, outcomes | Human-in-the-loop dashboard, Feedback Audit table, Enriched Campaign table, Pega Customer Decision Hub, Agent 8 Outcome Capture, Agent 9 Back Office |

**Core flow (the first milestone):**

```
scored account
     |
 [3 Researcher] -> [4 Paradox Triage] --confirmed--> [5 Offer Architect] -> [6 Compliance] -> [7 Prioritizer] -> human review
                          |
                       paradox
                          v
                   Exception Exit (suppress campaign, route to Service Recovery)
```

**Scope decision:** build Agents 3-7, the orchestrator and the human-in-the-loop dashboard first. Agents 1, 2, 8 and 9
are stretch goals, because they are mostly plumbing once the core works.

## 2. Build plan

Realistic pace: **about 6-8 hours a week**, roughly 5 months in total, with a working end-to-end slice by about week 8.

| Phase | What | Main learning curve |
|---|---|---|
| 0 | Foundations: repo, `AccountState`, LLM layer, agent contracts | State design |
| 1 | Synthetic data with planted truth (paradox cases, margin violations), policies, offer catalog, Outcomes KB | Realistic simulation, leakage control |
| 2 | Churn and acquisition models, MLflow, monthly scores | Familiar ground; sets up the feedback loop |
| 3 | Agent 3 + a two-node LangGraph | Tool calling, text-to-SQL safety, graph basics |
| 4 | Agents 4, 6, 5, 7 (in that order) | RAG quality, reasoning prompts, evaluating decisions |
| 5 | Full orchestration: conditional edges, retries, checkpointing | Routing, persistence, idempotency |
| 6 | Human-in-the-loop dashboard, audit table, Pega mock, Agents 8-9 | HITL patterns, integration contracts |
| 7 | Sense layer: Agent 2 (Whisper, PII masking) and Agent 1 | PII handling |
| 8 | Closing the loops: rejections to Agent 4, outcomes to the KB and retraining | Feedback design, drift |
| 9 | Hardening: guardrails, tracing, CI evals, scale test, enterprise mapping | Production readiness |

## 3. Decisions made today

1. **Provider-agnostic LLM layer.** Agents call one client; provider and model come from settings, with per-agent overrides.
2. **Local-first.** Ollama (`qwen3:8b`, thinking off) for development. The 100K-token API key is a scarce resource used
   for a few targeted runs, with a hard limit at 90,000 tokens.
3. **Deterministic first, LLM only for judgment.** Margin math, price lookup, ranking arithmetic, SQL safety and ID
   validation are code. The model selects, interprets, explains and retrieves.
4. **Typed state.** One Pydantic `AccountState`; each agent owns exactly one section.
5. **Planted ground truth instead of LLM judges** for evaluating agents. Exact, repeatable, zero tokens.
6. **Separation from the employer.** Personal machine, synthetic data, invented prices and policies.

Full detail is in `docs/DECISIONS.md` and `docs/AGENT_CONTRACTS.md`.

## 4. Your hardware and benchmark

**Laptop:** Lenovo Yoga 7 2-in-1, Core Ultra 7 258V, 32 GB LPDDR5X, Intel Arc 140V integrated graphics
(shared memory, no dedicated VRAM), Windows. Ollama runs on CPU here.

| Run | Total | Output tokens | Generation | Prompt processing |
|---|---|---|---|---|
| Thinking on | 75.6 s | 566 | 7.71 tok/s | 31.96 tok/s |
| Thinking off | 15.5 s | 126 | 9.59 tok/s | 32.21 tok/s |

Turning thinking off cut the same prompt from about 76 s to about 15 s. **Prompt processing (about 32 tok/s) is the
real bottleneck**, because the agents send long prompts. The cell below estimates the time for one account.

In [ ]:
# Measured on your laptop: qwen3:8b, thinking off, CPU inference
PROMPT_TPS = 32.21   # prompt processing, tokens per second
GEN_TPS = 9.59       # generation, tokens per second

# (prompt tokens, output tokens) per agent.
# Prompt budgets come from docs/AGENT_CONTRACTS.md; output sizes are my assumptions.
agents = {
    "3 Researcher":      (400, 150),
    "4 Paradox Triage":  (1000, 200),
    "5 Offer Architect": (700, 200),
    "6 Compliance":      (600, 150),
    "7 Prioritizer":     (500, 150),
}

print(f"{'agent':<20}{'prompt':>8}{'output':>8}{'seconds':>9}")
seconds = {}
for name, (p, o) in agents.items():
    seconds[name] = p / PROMPT_TPS + o / GEN_TPS
    print(f"{name:<20}{p:>8}{o:>8}{seconds[name]:>9.1f}")

total_s = sum(seconds.values())
print(f"\nOne account, all five agents (no cache, no retries): {total_s:.0f} s = {total_s / 60:.1f} min")

# Paradox accounts leave after Agent 4, so Agents 5-7 are skipped for them.
paradox_share = 0.15  # assumption
names = list(agents)
early = sum(seconds[n] for n in names[:2])
late = sum(seconds[n] for n in names[2:])
avg_s = early + (1 - paradox_share) * late
print(f"Average with {paradox_share:.0%} paradox early exits: {avg_s:.0f} s per account")

for n in (10, 50, 300):
    print(f"{n:>4} accounts: about {n * avg_s / 3600:.1f} hours")

**Reading this:** local runs are fine for developing one agent at a time and for small evaluation sets, but a
full-pipeline run over hundreds of accounts takes many hours on this machine. Levers for later: cache everything,
run the cheap `llama3.2:3b` for plumbing tests, keep prompts short with static instructions first (so Ollama reuses the
cached prefix), and keep deterministic steps out of the LLM.

## 5. Token budget for the 100K API key

The token meter refuses any API call that would push usage past **90,000** tokens (the 10,000 gap absorbs estimation error).
Usage is stored in `data/token_ledger.json`, and identical calls are served from `data/llm_cache/` for free.

In [ ]:
HARD_LIMIT = 90_000

per_account = sum(p + o for p, o in agents.values())
print(f"About {per_account:,} tokens per account for one full pass")
print(f"About {HARD_LIMIT // per_account} accounts fit under the {HARD_LIMIT:,}-token hard limit (no retries, no debugging)")
print("\nPlanned use of the API budget:")
for label, tokens in [("Agent 4 benchmark, local vs API (about 25 cases)", 40_000),
                      ("One polished demo run (about 5 accounts)", 45_000),
                      ("Reserve for debugging", 15_000)]:
    print(f"  {label:<52}{tokens:>8,}")

## 6. Setup recap (Windows, personal laptop)

**Ollama models**

```powershell
ollama pull qwen3:8b
ollama pull nomic-embed-text      # embeddings for the RAG knowledge bases later
```

**Python environment: option A, uv (recommended)**

```powershell
pip install uv                    # or: winget install astral-sh.uv
uv sync --extra dev
copy .env.example .env
uv run pytest
uv run python scripts\smoke_test.py
```

**Python environment: option B, conda**

```powershell
conda create -n fiberpulse python=3.11 -y
conda activate fiberpulse
pip install -e ".[dev]"
copy .env.example .env
pytest
python scripts\smoke_test.py
```

**Notebooks in VS Code:** install the Python and Jupyter extensions, then pick the project interpreter with
`Ctrl+Shift+P` then "Python: Select Interpreter". If VS Code asks for `ipykernel`, accept (or `pip install ipykernel`).

## 7. Phase 0 code tour

| File | Role |
|---|---|
| `src/fiberpulse/state.py` | `AccountState` plus the typed output of every agent |
| `src/fiberpulse/config.py` | Settings, per-agent provider routing, the synthetic 60% margin floor |
| `src/fiberpulse/llm.py` | `LLMClient`: Ollama or Anthropic, structured output with retries, disk cache, token ledger, hard API budget |
| `scripts/smoke_test.py` | End-to-end check of packages, Ollama, a plain call, a structured call, the ledger |
| `tests/` | State and LLM-layer tests using a fake model (no tokens spent) |

The cell below builds one `AccountState` by hand and shows the **paradox path**: the account leaves the pipeline,
and anything produced for Agents 5-7 is cleared.

In [ ]:
from fiberpulse.state import (
    AccountContext, AccountState, ModelScore, SignalType, TriageDecision, Verdict,
)

state = AccountState(
    account_id="ACC-000123",
    score=ModelScore(signal_type=SignalType.CHURN, score=0.82, model_version="churn-v1", scored_month="2026-10"),
)

# Invented example values
state.context = AccountContext(
    tenure_months=26, plan_name="Fiber Business 500", mrc_aed=1200, addon_count=2,
    open_complaints_90d=0, days_since_last_upsell=21, bill_trend_pct_3m=4.0, call_intent="positive",
)

state.triage = TriageDecision(
    verdict=Verdict.PARADOX, confidence=0.8,
    reasoning="A recent upsell and a positive support call contradict the high churn score.",
)
state.exit_with_exception(state.triage.reasoning)

print("idempotency key:", state.idempotency_key)
print("status:", state.status.value)
print("reason:", state.exception_reason)
print("offers / ranking cleared:", state.offers, state.ranking)

## 8. LLM client demo (optional, needs Ollama running)

This cell sends one tiny prompt through `LLMClient`. Run it twice: the second time the answer comes from the disk cache,
so it spends nothing and returns instantly. If Ollama is not running, the cell prints a message instead of failing.

In [ ]:
import dataclasses
from fiberpulse.config import Settings
from fiberpulse.llm import LLMClient

settings = dataclasses.replace(Settings.load(root / ".env"), data_dir=root / "data")
print("agent -> provider:", {a: settings.provider_for(a) for a in ("context_researcher", "paradox_triage")})

try:
    client = LLMClient(settings)
    result = client.invoke("context_researcher", [("human", "Reply with the single word: ready")], max_tokens=20)
    print("reply:", result.text.strip(), "|", "cached" if result.cached else f"live, {result.duration_s:.1f}s")
    print()
    print(client.ledger.summary())
except Exception as exc:
    print("Skipped (Ollama not running, or packages not installed):", type(exc).__name__, "-", exc)

## 9. Lessons from day 1

- **Thinking mode matters a lot on CPU.** Qwen3 with thinking on spent most of its tokens reasoning; off was about 5x faster
  for the same prompt. Set `OLLAMA_THINK=false`.
- **Prompt length drives latency**, not just output length. Budget prompts per agent.
- **`py -3` can fail even when Python exists.** The Windows `py` launcher only sees python.org installs; a conda Python
  is not registered with it. Use conda directly, or use `uv`, which can fetch Python itself.
- **A small local model can challenge a score.** In the first test, `qwen3:8b` questioned a 0.82 churn score when the
  customer had just upgraded and had a happy call. That prompt was easy; real paradox cases will be subtler, and the
  agent must commit to a verdict, not just hedge.
- **The token meter exists because the API budget is tiny.** About 22 accounts of full-pipeline calls fit in it.

**Not verified yet:** whether `pytest` passes on your machine (my environment used a stand-in), whether
`reasoning=False` is honoured by your installed `langchain-ollama` (upgrade it if thinking is still on),
and that your key can use the default API model `claude-haiku-5-5`.

## 10. Next: Phase 1, the synthetic data

Build a generator (default about 20K accounts) that writes:

- Tables: Customer_360, SR_Complaint, Addons, bills, Call_Intent_Signals, External_Enrichment
- Hidden churn and acquisition drivers, so models have real signal to learn
- Planted ground truth: about 25 paradox and about 25 confirmed accounts, offers below the margin floor, policy-violating offers
- Documents: policy set (including the margin floor), offer catalog with costs, Customer Outcomes knowledge base
- Synthetic call transcripts with fake PII, for the Voice Intelligence agent later

**Open items**

- Final project name (Project Axon, Project Maestro, FiberMesh AI, Project Sentinel; none chosen)
- Account count for the generator (default 20K)
- Whether to port Phases 5+ to Databricks Free Edition